# nb125 — Papyrus megafetch + PXR-relevant target activity dump (v2)

Filter Papyrus++ (~60M curated activity records) to the constellation of targets that share ligand-recognition motifs with PXR.

Targets: nuclear receptors (PXR/CAR/VDR/PPARs/LXRs/FXR/RXRs/RAR/ER/AR/GR/MR/TR/HNF4/ERR), P450s (CYP3A4, 2C9, 2C19, 1A2, 2D6, 2E1, 3A5, 2B6), drug transporters (MDR1/BCRP/MRP2/OATPs), xenosensor (AhR), and HSA (lipophilicity proxy).

Expected output: ~500k-2M activity records, saved as parquet for downstream Chemprop multi-target pretraining (nb127).

In [ ]:
import subprocess, sys, os, time
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'papyrus-scripts', 'rdkit', 'pystow'], check=False)
print('install done')

In [ ]:
# Download Papyrus++ filtered dataset
from papyrus_scripts import download_papyrus
print('Downloading Papyrus++ 05.7 ...')
download_papyrus(version='05.7', only_pp=True, structures=False, descriptors=None)
print('download complete')

In [ ]:
# Target list (UniProt accessions)
PXR_RELATED_TARGETS = {
    # nuclear receptors
    'PXR':       'O75469',
    'CAR':       'Q14994',
    'VDR':       'P11473',
    'FXR_NR1H4': 'Q96RI1',
    'LXRa':      'Q13133',
    'LXRb':      'P55055',
    'PPARa':     'Q07869', 'PPARg': 'P37231', 'PPARd': 'Q03181',
    'RARa':      'P10276', 'RARb': 'P10826',  'RARg': 'P13631',
    'RXRa':      'P19793', 'RXRb': 'P28702',  'RXRg': 'P48443',
    'AR':        'P10275', 'ER_a': 'P03372',  'ER_b': 'Q92731',
    'GR':        'P04150', 'MR':   'P08235',
    'TRa':       'P10827', 'TRb':  'P10828',
    'HNF4a':     'P41235',
    'ERRa':      'P11474', 'ERRb': 'O95718',  'ERRg': 'P62508',
    # P450s
    'CYP3A4':    'P08684', 'CYP2C9': 'P11712', 'CYP2C19': 'P33261',
    'CYP1A2':    'P05177', 'CYP2D6': 'P10635', 'CYP2E1':  'P05181',
    'CYP3A5':    'P20815', 'CYP2B6': 'P20813',
    # transporters
    'MDR1':      'P08183', 'BCRP':   'Q9UNQ0', 'MRP2':    'Q92887',
    'OATP1B1':   'Q9Y6L6', 'OATP1B3': 'Q9NPD5',
    # xenosensor + reference
    'AhR':       'P35869', 'HSA':    'P02768',
}
target_uniprots = list(set(PXR_RELATED_TARGETS.values()))
print(f'Targets: {len(PXR_RELATED_TARGETS)} ({len(target_uniprots)} unique UniProts)')

In [ ]:
# Stream + filter Papyrus++
from papyrus_scripts.reader import read_papyrus
from papyrus_scripts.preprocess import keep_accession
import pandas as pd, time

t0 = time.time()
filtered_chunks = []
total_seen = 0
for i, chunk in enumerate(read_papyrus(version='05.7', plusplus=True, is3d=False, chunksize=100_000)):
    sub = keep_accession(chunk, target_uniprots)
    total_seen += len(chunk)
    if len(sub) > 0:
        filtered_chunks.append(sub)
    if i % 5 == 0:
        kept = sum(len(c) for c in filtered_chunks)
        print(f'  chunk {i}: scanned {total_seen:,}, kept {kept:,}  ({time.time()-t0:.0f}s)')

papy = pd.concat(filtered_chunks, ignore_index=True)
print(f'\nFinal: {len(papy):,} rows x {papy.shape[1]} cols  in {time.time()-t0:.0f}s')
print(f'Columns: {list(papy.columns)[:25]}')
print(f'Targets covered: {papy["accession"].nunique()}')

In [ ]:
# Save filtered data + summaries
from pathlib import Path
out_dir = Path('/kaggle/working/papyrus_pxr_megafetch')
out_dir.mkdir(parents=True, exist_ok=True)
papy.to_parquet(out_dir / 'papyrus_pxr_related_filtered.parquet', index=False)
print(f'Saved {len(papy):,} rows')

# per-target summary
value_col = 'pchembl_value_Mean' if 'pchembl_value_Mean' in papy.columns else (
    'pchembl_value_StdDev' if 'pchembl_value_StdDev' in papy.columns else papy.columns[-1])
smiles_col = 'SMILES' if 'SMILES' in papy.columns else (
    'SMILES_Stripped' if 'SMILES_Stripped' in papy.columns else 'connectivity')
print(f'value_col={value_col}, smiles_col={smiles_col}')

summary = papy.groupby('accession').agg(
    n_records=(value_col, 'count'),
    mean_pchembl=(value_col, 'mean'),
    std_pchembl=(value_col, 'std'),
).sort_values('n_records', ascending=False)
summary.to_csv(out_dir / 'target_summary.csv')
print(summary.head(20))

In [ ]:
# Build compound x target wide matrix for multi-target pretraining
wide = papy.pivot_table(index=smiles_col, columns='accession', values=value_col, aggfunc='median')
print(f'Wide pivot: {wide.shape}  (compounds x targets)')
wide.reset_index().to_parquet(out_dir / 'papyrus_wide_compound_x_target.parquet', index=False)
print('Saved wide pivot')
print('All outputs in /kaggle/working/papyrus_pxr_megafetch/')